# Phase 2 · Experiment 2D — Language Comparison

### Google Colab notebook (independent)

Research question: **do English and Vietnamese differ in sink behaviour under matched conditions?** Using Phase 1’s sentence-aligned pairs, we compare average sink score, layer profiles and head profiles, with a **paired** statistical test. Because Vietnamese sentences tokenise longer (Phase 1 finding #4), we rely on the length-robust `mean_from_k` metric and report whether any language difference tracks the per-pair length delta. We determine **whether** a measurable difference exists — not why (that is Phase 3).

**Runtime:** CPU is fine — this notebook only reads Phase 1 attention.

**Prerequisites**
- Phase 1 must have been run with `USE_DRIVE=True`, so `attention_sink_data/` is in your Drive project folder.
- `phase2_utils.py` must be uploaded into the same Drive project folder (upload once; it persists).
- No model download needed. Reuses Experiment 2A’s baseline if present, else recomputes it.

This notebook is self-contained: it can be rerun on its own without executing the other experiments.

---

## 0. Setup

In [ ]:
# --- Colab environment setup -------------------------------------------------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', *['scipy', 'pyarrow']], check=True)

print('In Colab:', IN_COLAB)
print('Analysis-only notebook: no GPU required.')


In [ ]:
# --- Storage + phase2_utils bootstrap ---------------------------------------
# Point at the SAME Drive project folder Phase 1 used, so Phase 1's raw
# attention and this project's phase2_utils.py are both visible.
USE_DRIVE = True   # must match the Phase 1 setting

import sys
from pathlib import Path
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/attention_sink_project')
elif IN_COLAB:
    BASE = Path('/content/attention_sink_project')
else:
    BASE = Path('.')
BASE.mkdir(parents=True, exist_ok=True)

DATA_ROOT    = str(BASE / 'attention_sink_data')          # Phase 1 output
RESULTS_ROOT = str(BASE / 'results' / 'phase2')           # Phase 2 output
Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)

# Locate phase2_utils.py (upload it into BASE once; it persists on Drive).
for c in [BASE, Path('/content'), Path('.')]:
    if (Path(c) / 'phase2_utils.py').exists():
        sys.path.insert(0, str(c)); break
try:
    import phase2_utils as U
    print('phase2_utils loaded from', U.__file__)
except ModuleNotFoundError:
    raise SystemExit('Place phase2_utils.py in ' + str(BASE) + ' (or /content) and re-run this cell.')

print('Phase 1 data :', DATA_ROOT)
print('Phase 2 out  :', RESULTS_ROOT)


## 1. Configuration

In [ ]:
from dataclasses import dataclass, asdict
import json, numpy as np, pandas as pd

@dataclass
class Config2D:
    k: int = 4
    sink_threshold: float = 0.30
    seed: int = 20240517

cfg = Config2D()
U.set_reproducibility(cfg.seed)
EXP = U.experiment_dir(RESULTS_ROOT, 'experiment2D')
logger = U.get_logger('2D', log_file=str(EXP / 'run.log'))

## 2. Get the sink tensor (reuse 2A baseline if available, else recompute)

In [ ]:
meta = pd.read_csv(Path(DATA_ROOT) / 'metadata.csv')
base = U.load_baseline(Path(RESULTS_ROOT) / 'experiment2A')
if base is not None and list(base['prompt_ids']) == meta['prompt_id'].tolist():
    tensor = base['tensor']; prompt_ids = base['prompt_ids']
    logger.info('Reusing Experiment 2A baseline tensor %s', tensor.shape)
else:
    prompt_ids = meta['prompt_id'].tolist()
    tensor = U.compute_sink_tensor(DATA_ROOT, prompt_ids, k=cfg.k)   # independent recompute
    logger.info('Recomputed sink tensor %s (2A artefact not reusable)', tensor.shape)
meta_idx = meta.set_index('prompt_id')
lang = np.array([meta_idx.loc[p, 'language'] for p in prompt_ids])
print('tensor:', tensor.shape, '| en:', int((lang=='en').sum()), 'vi:', int((lang=='vi').sum()))

## 3. Language profiles + per-pair paired globals

In [ ]:
prompt_global = tensor.mean(axis=(1, 2))                 # [P]
en_mask, vi_mask = (lang == 'en'), (lang == 'vi')

layer_prof = {'en': tensor[en_mask].mean(axis=(0, 2)), 'vi': tensor[vi_mask].mean(axis=(0, 2))}
layer_std  = {'en': tensor[en_mask].mean(axis=2).std(axis=0), 'vi': tensor[vi_mask].mean(axis=2).std(axis=0)}
head_prof  = {'en': tensor[en_mask].mean(axis=(0, 1)), 'vi': tensor[vi_mask].mean(axis=(0, 1))}

# per-prompt frame -> pivot to matched pairs
pp = pd.DataFrame({'prompt_id': prompt_ids, 'language': lang,
                   'pair_id': [meta_idx.loc[p, 'pair_id'] for p in prompt_ids],
                   'seq_len': [meta_idx.loc[p, 'seq_len'] for p in prompt_ids],
                   'global': prompt_global})
pp.to_csv(EXP / 'per_prompt.csv', index=False)
piv = pp.pivot_table(index='pair_id', columns='language', values='global').dropna()
len_piv = pp.pivot_table(index='pair_id', columns='language', values='seq_len').dropna()
print('matched pairs:', len(piv))

## 4. Paired statistics + length-confound check (whether, not why)

In [ ]:
test = U.paired_test(piv['en'].values, piv['vi'].values)

# does the sink difference track the tokenisation length difference?
sink_diff = (piv['vi'] - piv['en']).reindex(len_piv.index).values
len_diff = (len_piv['vi'] - len_piv['en']).values
corr = float(np.corrcoef(sink_diff, len_diff)[0, 1]) if len(sink_diff) > 2 else float('nan')

result = {'metric': 'mean_from_k', 'n_pairs': int(len(piv)),
          'en_mean': float(piv['en'].mean()), 'vi_mean': float(piv['vi'].mean()),
          'paired_test': test,
          'mean_len_delta_vi_minus_en': float(len_diff.mean()),
          'corr_sinkdiff_lendiff': corr}
with open(EXP / 'language_paired_stats.json', 'w') as f:
    json.dump(result, f, indent=2)
pd.DataFrame([{ 'n_pairs': result['n_pairs'], 'en_mean': result['en_mean'],
               'vi_mean': result['vi_mean'], 'mean_diff': test['mean_diff'],
               'cohens_d_paired': test['cohens_d_paired'], 'test': test['test'],
               'corr_sinkdiff_lendiff': corr}]).to_csv(EXP / 'language_paired_stats.csv', index=False)
print(json.dumps(result, indent=2))

## 5. Plots

In [ ]:
U.plot_layer_profile(layer_prof, EXP / 'figures', 'layer_profile_by_language.png',
                     stds=layer_std, title='Layer profile by language (mean_from_k)')
U.plot_head_profile(head_prof, EXP / 'figures', 'head_profile_by_language.png',
                    title='Head profile by language (mean_from_k)')
U.plot_language_paired(pp, EXP / 'figures', 'global_sink_paired.png', value='global')

d = test['cohens_d_paired']
print('Read-out (whether a difference exists):')
print('  en mean=%.4f  vi mean=%.4f  paired d=%+.3f' % (result['en_mean'], result['vi_mean'], d))
print('  effect size:', 'negligible' if abs(d) < 0.2 else 'small' if abs(d) < 0.5 else 'medium+' )
print('  corr(sink diff, length diff) = %.2f' % corr,
      '-> difference may be a length effect' if abs(corr) > 0.5 else '-> not strongly length-linked')

## Download results

In [ ]:
# --- Download this experiment's outputs -------------------------------------
import shutil
exp = Path(RESULTS_ROOT) / 'experiment2D'
zp = shutil.make_archive(str(Path('/content' if IN_COLAB else '.') / ('experiment2D_outputs')), 'zip', exp)
print('Bundled:', zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)
